# E1 multi-task grader — pre-training gates, then (separately) seeds 42 → 123 → 2026

**Model.** P unchanged (ImageNet ConvNeXt-Tiny → average pooling → LayerNorm → CORN) plus one 1×1 convolution on the final
16×16×768 feature map that predicts the frozen Stage-4 lesion maps (MA, HE, EX, SE) pooled to 16×16. 3,076 added parameters.
**Loss.** L = L_CORN + 1.0 · L_lesion (mean sigmoid cross-entropy against the soft teacher targets). λ = 1 is fixed.
**Protocol.** P's, unchanged: batch 2, AdamW 1e-4 / wd 0.05, weighted CORN, ≤ 50 epochs, val-QWK early stopping, mixed precision,
P's augmentation and epoch order, seeds 42 / 123 / 2026, BEST by validation QWK. No vessel supervision. No IDRiD.

**Cells 4–7 are the gates. They train nothing.**
- Cell 4 — Gate 3, targets: statistics and hard checks of all 2,921 training targets (label-free); gives the bias prior.
- Cell 5 — Gate 1, P parity: E1's grading path with each pinned P BEST checkpoint reproduces P on the 730 validation images
  (float32: 1e-4; mixed_float16 vs the stored P tables: logits 0.05, probabilities 0.01; grades identical), and equals P at initialisation.
- Cell 6 — Gate 2, log aliases: the actual Keras log keys of the two-output model on this runtime, and that the checkpoint
  state and the stored history receive `val_QWK`. One epoch on a 5-image synthetic bundle with random weights — mechanics only.
- Cell 7 — gradient check on one real batch: each loss alone reaches every encoder variable.

A failed gate stops here: no tolerance is loosened and nothing is trained.
**Cell 8 trains** and is switched off (`RUN_TRAINING = False`). It refuses to start unless the three gates are recorded as passed.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / IDRiD / Stage-4 weights needed.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad, arch1_train as at
import e1_data as ed, e1_model as em, e1_train as et, e1_gates as eg
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] INPUTS: Stage-4 teacher identity, bundle, P protocol, pinned P BEST checkpoints ====
EXP = colab_config.DRIVE.experiments_root
DRIVE_ROOT = colab_config.DRIVE.project_root
EXPECTED_MODEL_SHA = "cb5fc7a8d370af7d2ae191cadaebdde858f71820d147fb76f852b757766f8ad8"   # gate-passed Stage-4 v2 (§47/§48)
GEN4 = cache.stage4_generation_id(EXPECTED_MODEL_SHA, len(v2cfg.STAGE4_V2A_CLASSES))
assert GEN4 == "s4v2-cb5fc7a8d370-K4", GEN4
BUNDLE_ID = cache.bundle_id(v2cfg.STAGE2_RGB_GENERATION, v2cfg.STAGE3_GENERATION, GEN4)
SIX_RUN_DIR = msr.experiment_root(posixpath.join(EXP, "ImprovedTraining"), "improved_multiseed_2026_09")
PREREG, _ = msr.load_and_verify_preregistration(posixpath.join(SIX_RUN_DIR, msr.PREREGISTRATION_FILENAME))
CLASS_WEIGHTS = list(PREREG["class_weights"])
at.assert_protocol(PREREG, CLASS_WEIGHTS)
SEEDS = et.SEEDS
assert SEEDS == (42, 123, 2026) and et.LAMBDA == 1.0 and at.EMA == "none"
assert em.LESION_CLASSES == ("MA", "HE", "EX", "SE") and em.ADDED_PARAMETERS == 3076 and ed.GRID == 16
# The pinned P BEST checkpoints (SHA-256) are the ones recorded in the frozen evaluation lock file (record §60).
_PINS = json.load(open(posixpath.join(REPO_DIR, "idrid_grading_protocol.json"), encoding="utf-8"))["checkpoints"]
P_PATHS = {s: posixpath.join(DRIVE_ROOT, _PINS[f"p_seed{s}"]["path"]) for s in SEEDS}
P_SHA256 = {s: _PINS[f"p_seed{s}"]["sha256"] for s in SEEDS}
PRETRAINED = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))["local_path"]
assert cache.sha256_file(PRETRAINED) == pl.WEIGHTS_SHA256 == _PINS["convnext_pretrained"]["sha256"]
GATE_DIR = posixpath.join(EXP, et.EXPERIMENT, "gates")
RUN_DIRS = {s: et.run_dir_for(EXP, EXPECTED_MODEL_SHA, s) for s in SEEDS}
print("bundle", BUNDLE_ID, "| gates ->", GATE_DIR)
for _s in SEEDS:
    print(f"  seed {_s}: {et.seed_state(RUN_DIRS[_s]):<11} {RUN_DIRS[_s]} | P BEST {P_SHA256[_s][:12]}")

In [ ]:
# ==== [3] VERIFY THE BUNDLE ON DRIVE, STAGE THE RGB + LESION CACHES LOCALLY (no vessel files), VERIFY ====
DRIVE_ROOTS = dict(ad.DEFAULT_ROOTS)
LOCAL_ROOTS = {k: posixpath.join("/content/cache_v2_e1", os.path.basename(v)) for k, v in DRIVE_ROOTS.items()}
ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256, expected_stage4_sha256=EXPECTED_MODEL_SHA)
ed.stage_inputs(BUNDLE_ID, DRIVE_ROOTS, LOCAL_ROOTS)
BUNDLE = ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256,
                        expected_stage4_sha256=EXPECTED_MODEL_SHA, roots=LOCAL_ROOTS)
ed.require_inputs(BUNDLE, BUNDLE.population)
assert BUNDLE.stage4_sha256 == EXPECTED_MODEL_SHA and BUNDLE.stage4_generation == GEN4
_tr, _va, _split_sha = msr.verify_split()
_empty = set(v2cfg.APTOS_EMPTY_FOV_IDS)
assert _split_sha == v2cfg.SPLIT_SHA256
assert list(BUNDLE.train_ids) == [str(i) for i, _ in _tr if str(i) not in _empty] and len(BUNDLE.train_ids) == 2921
assert list(BUNDLE.val_ids) == [str(i) for i, _ in _va if str(i) not in _empty] and len(BUNDLE.val_ids) == 730
assert not os.path.isdir(BUNDLE.dirs["stage3_cache_v2"]), "the vessel cache must not be staged for E1"
print(json.dumps(BUNDLE.describe(), indent=1))

In [ ]:
# ==== [4] GATE 3 -- TARGETS: all 2,921 training targets (label-free). Trains nothing. ====
GATE_TARGETS = eg.targets(BUNDLE, out_dir=GATE_DIR)
print(json.dumps(GATE_TARGETS["statistics"], indent=1))
assert GATE_TARGETS["PASS"], f"TARGET GATE FAILED -- stop. {GATE_TARGETS['failures']}"
LESION_PRIOR = GATE_TARGETS["lesion_prior"]
print("lesion prior (MA, HE, EX, SE):", [round(p, 5) for p in LESION_PRIOR],
      "| bias:", [round(b, 4) for b in GATE_TARGETS["lesion_prior_logits"]])

In [ ]:
# ==== [5] GATE 1 -- P PARITY on the 730 validation images, both tiers. Trains nothing. ====
GATE_PARITY = eg.p_parity(BUNDLE, P_PATHS, PRETRAINED, LESION_PRIOR, policies=("float32", "mixed_float16"),
                          stored_root=EXP, expected_sha256=P_SHA256, out_dir=GATE_DIR)
print(json.dumps({"PASS": GATE_PARITY["PASS"], "failures": GATE_PARITY["failures"],
                  "initialisation": GATE_PARITY["initialisation"],
                  "float32": {s: r["e1_vs_live_p"] for s, r in GATE_PARITY["tiers"]["float32"].items()},
                  "mixed_float16": {s: r.get("e1_vs_stored_p_best") for s, r in GATE_PARITY["tiers"]["mixed_float16"].items()},
                  "environment": GATE_PARITY["environment"]}, indent=1, default=float))
assert GATE_PARITY["PASS"], "P PARITY FAILED -- stop. Do not loosen tolerances. Do not train."


In [ ]:
# ==== [5D] GATE 1 DIAGNOSTIC -- no pass / fail, no rule or tolerance changed. Trains nothing. ====
# P recomputed under mixed_float16 on this runtime vs P's STORED validation logits, per image; and, independently,
# E1 vs that recomputed P. Runs in a session where cells 1-4 have run (cell 5's assertion does not block it).
import importlib
subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
importlib.reload(eg)
DIAG = eg.p_parity_diagnostic(BUNDLE, P_PATHS, PRETRAINED, LESION_PRIOR, EXP, expected_sha256=P_SHA256, out_dir=GATE_DIR)
for _s, _r in DIAG["seeds"].items():
    _d = _r["recomputed_p_vs_stored_p"]
    print(f"\n===== seed {_s} | stored logit range {_r['stored_logit_range']} | stored table consistent {_r['stored_table_internally_consistent']}")
    print("logit |diff| (all 730 x 4):", _d["logit_abs_diff"])
    print("logit |diff| per-image max:", _d["logit_abs_diff_per_image_max"])
    print("probability |diff|:", _d["probability_abs_diff"])
    print("decoded grades differ:", _d["decoded_grades_differ"])
    print("worst image:", json.dumps(_d["worst_image"], indent=1))
    print("saturation:", json.dumps(_d["saturation"], indent=1))
    print("shift:", _d["shift"])
    print("E1 vs recomputed P (same runtime):", _r["e1_vs_recomputed_p_same_runtime"])
    print("P at batch 2 vs batch 8 (same policy):", _r["recomputed_p_other_batch_size_vs_this_one_same_policy"])
    print("P float32 vs stored:", _r["recomputed_p_float32_vs_stored_p"])
    print("P mixed vs P float32:", _r["recomputed_p_policy_vs_recomputed_p_float32"])
print("\nenvironment:", DIAG["environment"], "| files ->", GATE_DIR)

In [ ]:
# ==== [6] GATE 2 -- LOG ALIASES on this runtime's Keras. Synthetic 5-image bundle, random weights, ONE epoch: mechanics only. ====
import shutil, keras
sys.path.insert(0, posixpath.join(REPO_DIR, "tests"))
import v2_bundle_fixture as fx
_work = "/content/e1_alias_gate"
shutil.rmtree(_work, ignore_errors=True)
_fx = fx.build(posixpath.join(_work, "data"))
_fx_bundle = ad.Arch1Bundle(expected_bundle_id=_fx["bundle"]["bundle_id"], expected_split_sha256=v2cfg.SPLIT_SHA256,
                            expected_stage4_sha256=fx.MODEL_SHA, roots=_fx["roots"], expected_population=5)
keras.mixed_precision.set_global_policy("float32")
keras.utils.set_random_seed(7)
_ref = keras.applications.ConvNeXtTiny(include_top=False, weights=None, include_preprocessing=True, pooling="avg",
                                       input_shape=(512, 512, 3), name=pl.BACKBONE_NAME)       # RANDOM weights
GATE_ALIAS = eg.log_alias(_work, _fx_bundle, dict(fx.TRAIN + fx.VAL), _ref, CLASS_WEIGHTS, mixed_precision=True,
                          repo_dir=REPO_DIR, out_dir=GATE_DIR)
print("Keras", keras.__version__, "| raw log keys:", GATE_ALIAS["raw_keras_log_keys"])
print("after the alias callback:", GATE_ALIAS["keys_after_alias"])
print("callback order:", GATE_ALIAS["callback_order"])
print("stored history row:", GATE_ALIAS["stored_history_row"])
assert GATE_ALIAS["PASS"], f"LOG ALIAS GATE FAILED -- stop. Do not train. {GATE_ALIAS['failures']}"
del _ref

In [ ]:
# ==== [7] GRADIENT CHECK on one real training batch (ImageNet-initialised E1, the run's precision). Trains nothing. ====
tf.keras.backend.clear_session()
_reference, _ = pl.load_reference(PRETRAINED)
_model = et.build_compiled_model(SEEDS[0], LESION_PRIOR, _reference, CLASS_WEIGHTS)
_grade = {str(i): int(g) for i, g in list(_tr) + list(_va)}
_x, (_g, _t) = ed.make_epoch_sequence(BUNDLE, [(i, _grade[i]) for i in BUNDLE.train_ids[:2]], 0, SEEDS[0], 2, augment=True)[0]
GATE_GRADIENT = eg.gradient_report(_model, _x["rgb"], _g, _t, CLASS_WEIGHTS)
print(json.dumps(GATE_GRADIENT, indent=1), "\nparameters:", em.parameter_report(_model))
with open(posixpath.join(GATE_DIR, "gradient_check.json"), "w") as fh:
    json.dump(GATE_GRADIENT, fh, indent=1)
assert GATE_GRADIENT["PASS"], f"GRADIENT CHECK FAILED -- stop. {GATE_GRADIENT['failures']}"
del _model, _reference
GATES = {"p_parity": GATE_PARITY, "log_alias": GATE_ALIAS, "targets": GATE_TARGETS}
print("\nGATES:", {k: ("PASS" if v["PASS"] and v["official"] else "FAIL") for k, v in GATES.items()}, "| gradient check:",
      "PASS" if GATE_GRADIENT["PASS"] else "FAIL")

In [ ]:
# ==== [8] TRAIN + EVALUATE SEEDS 42 -> 123 -> 2026 (resumable) -- THIS TRAINS. Switched off. ====
# Run only after the gate results have been reported and the pre-run record is written.
RUN_TRAINING = False
assert RUN_TRAINING, "training is switched off: the gates are reported first"
assert all(g["PASS"] and g["official"] for g in GATES.values()) and GATE_GRADIENT["PASS"], "a gate has not passed"
tf.keras.backend.clear_session()
REFERENCE, _ = pl.load_reference(PRETRAINED)
RESULTS = et.run_sequence(EXP, BUNDLE, LESION_PRIOR, REFERENCE, CLASS_WEIGHTS, repo_dir=REPO_DIR,
                          staging_root="/content/e1_staging",
                          gates={k: {"PASS": v["PASS"] and v["official"], "timestamp_utc": v["timestamp_utc"]} for k, v in GATES.items()})